In [4]:
import requests
from bs4 import BeautifulSoup
import os
import json
import re

# Añade aquí la lista completa de las 50 URLs del mensaje anterior
urls_corpus = [
    "https://www.smogon.com/dex/sv/pokemon/great-tusk/",
    "https://www.smogon.com/dex/sv/pokemon/gholdengo/",
    "https://www.smogon.com/dex/sv/pokemon/kingambit/",
    "https://www.smogon.com/dex/sv/pokemon/iron-valiant/",
    "https://www.smogon.com/dex/sv/pokemon/roaring-moon/",
    "https://www.smogon.com/dex/sv/pokemon/dragapult/",
    "https://www.smogon.com/dex/sv/pokemon/ting-lu/",
    "https://www.smogon.com/dex/sv/pokemon/corviknight/",
    "https://www.smogon.com/dex/sv/pokemon/dondozo/",
    "https://www.smogon.com/dex/sv/pokemon/clodsire/",
    "https://www.smogon.com/dex/sv/pokemon/iron-treads/",
    "https://www.smogon.com/dex/sv/pokemon/amoonguss/",
    "https://www.smogon.com/dex/sv/pokemon/hatterene/",
    "https://www.smogon.com/dex/sv/pokemon/garganacl/",
    "https://www.smogon.com/dex/sv/pokemon/azumarill/",
    "https://www.smogon.com/dex/sv/pokemon/rotom-wash/",
    "https://www.smogon.com/dex/sv/pokemon/skeledirge/",
    "https://www.smogon.com/dex/sv/pokemon/volcarona/",
    "https://www.smogon.com/dex/sv/pokemon/baxcalibur/",
    "https://www.smogon.com/dex/sv/pokemon/ceruledge/",
    "https://www.smogon.com/dex/sv/pokemon/garchomp/",
    "https://www.smogon.com/dex/sv/pokemon/dragonite/",
    "https://www.smogon.com/dex/sv/pokemon/breloom/",
    "https://www.smogon.com/dex/sv/pokemon/cinderace/",
    "https://www.smogon.com/dex/sv/pokemon/greninja/",
    "https://www.smogon.com/dex/sv/pokemon/iron-moth/",
    "https://www.smogon.com/dex/sv/pokemon/iron-hands/",
    "https://www.smogon.com/dex/sv/pokemon/slowking-galar/",
    "https://www.smogon.com/dex/sv/pokemon/toxapex/",
    "https://www.smogon.com/dex/sv/pokemon/zapdos/",
    "https://www.smogon.com/dex/sv/pokemon/landorus-therian/",
    "https://www.smogon.com/dex/sv/pokemon/ogerpon-wellspring/",
    "https://www.smogon.com/dex/sv/pokemon/rillaboom/",
    "https://www.smogon.com/dex/sv/pokemon/sneasler/",
    "https://www.smogon.com/dex/sv/pokemon/zamazenta/",
    "https://www.smogon.com/dex/sv/pokemon/kyurem/",
    "https://www.smogon.com/dex/sv/pokemon/gliscor/",
    "https://www.smogon.com/dex/sv/pokemon/primarina/",
    "https://www.smogon.com/dex/sv/pokemon/clefable/",
    "https://www.smogon.com/dex/sv/pokemon/heatran/",
    "https://www.smogon.com/dex/sv/pokemon/torkoal/",
    "https://www.smogon.com/dex/sv/pokemon/walking-wake/",
    "https://www.smogon.com/dex/sv/pokemon/gouging-fire/",
    "https://www.smogon.com/dex/sv/pokemon/raging-bolt/",
    "https://www.smogon.com/dex/sv/pokemon/iron-boulder/",
    "https://www.smogon.com/dex/sv/pokemon/deoxys-speed/",
    "https://www.smogon.com/dex/sv/pokemon/meowscarada/",
    "https://www.smogon.com/dex/sv/pokemon/darkrai/",
    "https://www.smogon.com/dex/sv/pokemon/latios/",
    "https://www.smogon.com/dex/sv/pokemon/scizor/"
]

directorio_corpus = "../corpus"
# Si la carpeta no existe, la crea
if not os.path.exists(directorio_corpus):
    os.makedirs(directorio_corpus)
for url in urls_corpus:
    try:
        # Extraemos el nombre del Pokémon directamente de la URL
        # strip("/") quita la última barra si la hay, y split("/")[-1] coge la última palabra (ej: "great-tusk")
        nombre_pokemon = url.strip("/").split("/")[-1]
        
        respuesta = requests.get(url)
        html = respuesta.text
        
        if 'dexSettings = ' not in html:
            print(f"No se encontraron datos en la página para {nombre_pokemon}")
            continue
            
        texto_json = html.split('dexSettings = ', 1)[1].split('</script>')[0].strip()
        
        if texto_json.endswith(';'):
            texto_json = texto_json[:-1]
            
        data = json.loads(texto_json)
        
        pokemon_data = None
        for rpc in data.get('injectRpcs', []):
            if 'dump-pokemon' in rpc[0]:
                pokemon_data = rpc[1]
                break
                
        if not pokemon_data or 'strategies' not in pokemon_data or len(pokemon_data['strategies']) == 0:
            print(f"No hay estrategias para {nombre_pokemon}")
            continue
            
        html_estrategias = ""
        for strat in pokemon_data['strategies']:
            html_estrategias += strat.get('overview', '') + "\n"
            for moveset in strat.get('movesets', []):
                html_estrategias += moveset.get('description', '') + "\n"
            html_estrategias += strat.get('comments', '') + "\n"
            
        soup = BeautifulSoup(html_estrategias, 'html.parser')
        texto_limpio = soup.get_text(separator='\n\n', strip=True)
        
        if not texto_limpio.strip():
            print(f"Advertencia: El documento de {nombre_pokemon} está vacío")
            continue
            
        # Guardamos el documento con el nombre del Pokémon (ej: ../corpus/great-tusk.txt)
        nombre_fichero = f"{directorio_corpus}/{nombre_pokemon}.txt"
        with open(nombre_fichero, "w", encoding="utf-8") as f:
            f.write(texto_limpio)
            
        print(f"Documento {nombre_pokemon}.txt guardado con éxito en la carpeta corpus.")
        
    except Exception as e:
        print(f"Error al procesar a {nombre_pokemon}: {e}")

Documento great-tusk.txt guardado con éxito en la carpeta corpus.
Documento gholdengo.txt guardado con éxito en la carpeta corpus.
Documento kingambit.txt guardado con éxito en la carpeta corpus.
Documento iron-valiant.txt guardado con éxito en la carpeta corpus.
Documento roaring-moon.txt guardado con éxito en la carpeta corpus.
Documento dragapult.txt guardado con éxito en la carpeta corpus.
Documento ting-lu.txt guardado con éxito en la carpeta corpus.
Documento corviknight.txt guardado con éxito en la carpeta corpus.
Documento dondozo.txt guardado con éxito en la carpeta corpus.
Documento clodsire.txt guardado con éxito en la carpeta corpus.
Documento iron-treads.txt guardado con éxito en la carpeta corpus.
Documento amoonguss.txt guardado con éxito en la carpeta corpus.
Documento hatterene.txt guardado con éxito en la carpeta corpus.
Documento garganacl.txt guardado con éxito en la carpeta corpus.
Documento azumarill.txt guardado con éxito en la carpeta corpus.
Documento rotom-was